<a href="https://colab.research.google.com/github/rodrigologin0-cpu/Rodrigo-de-Souza-Lima/blob/main/Identifica%C3%A7%C3%A3o_de_Sistemas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# IDENTIFICAÇÃO DE SISTEMAS - MODELO ARX SISO / MISO
# Autor: Rodrigo Lima
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Union, Dict


In [ ]:
# ============================================================
# CONFIGURAÇÕES DO USUÁRIO
# ============================================================

ARQUIVO_DADOS = "dados.xlsx"   # Pode ser .xlsx ou .csv

COLUNA_SAIDA = "Pressao_Forno"

COLUNAS_ENTRADA = [
    "Damper"
    # Para MISO, exemplo:
    # "Damper",
    # "Porta",
    # "Demanda_Termica"
]

TS = 1.0  # tempo de amostragem em segundos

# Ordem do modelo ARX
NA = 2        # número de atrasos da saída
NB = 2        # número de coeficientes por entrada
NK = 1        # atraso puro da entrada em amostras

# Regularização opcional.
# Use 0.0 para mínimos quadrados puro.
# Valores pequenos como 1e-6, 1e-4 ou 1e-2 podem ajudar com dados ruidosos.
RIDGE_LAMBDA = 0.0

# Percentual inicial dos dados usado para treino
PERC_TREINO = 0.7


In [ ]:
# ============================================================
# ESTRUTURAS
# ============================================================

@dataclass
class ModeloARX:
    na: int
    nb: List[int]
    nk: List[int]
    theta: np.ndarray
    a: np.ndarray
    b: List[np.ndarray]
    ts: float
    nomes_entradas: List[str]
    nome_saida: str


In [ ]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def carregar_dados(caminho: str) -> pd.DataFrame:
    """
    Carrega dados de arquivo Excel ou CSV.
    """
    if caminho.lower().endswith(".xlsx") or caminho.lower().endswith(".xls"):
        df = pd.read_excel(caminho)
    elif caminho.lower().endswith(".csv"):
        df = pd.read_csv(caminho, sep=None, engine="python")
    else:
        raise ValueError("Formato não suportado. Use .xlsx, .xls ou .csv.")

    return df


def tratar_nb_nk(nb: Union[int, List[int]], nk: Union[int, List[int]], n_entradas: int):
    """
    Permite usar NB e NK como número único ou lista por entrada.
    """
    if isinstance(nb, int):
        nb = [nb] * n_entradas

    if isinstance(nk, int):
        nk = [nk] * n_entradas

    if len(nb) != n_entradas:
        raise ValueError("NB deve ter o mesmo tamanho do número de entradas.")

    if len(nk) != n_entradas:
        raise ValueError("NK deve ter o mesmo tamanho do número de entradas.")

    return nb, nk


def montar_matriz_arx(y: np.ndarray,
                      u: np.ndarray,
                      na: int,
                      nb: List[int],
                      nk: List[int]):
    """
    Monta a matriz de regressão Phi para o modelo ARX.

    Modelo:
    y(k) + a1*y(k-1) + ... + ana*y(k-na)
      = b1*u(k-nk) + b2*u(k-nk-1) + ... + erro

    Logo:
    y(k) = -a1*y(k-1) - ... - ana*y(k-na)
           + b1*u(k-nk) + b2*u(k-nk-1) + ...
    """

    n_amostras = len(y)
    n_entradas = u.shape[1]

    maior_atraso_saida = na
    maior_atraso_entrada = max([nk[j] + nb[j] - 1 for j in range(n_entradas)])
    k0 = max(maior_atraso_saida, maior_atraso_entrada)

    phi = []
    alvo = []

    for k in range(k0, n_amostras):
        linha = []

        # Termos da saída passada: -y(k-1), -y(k-2), ...
        for i in range(1, na + 1):
            linha.append(-y[k - i])

        # Termos das entradas passadas
        for j in range(n_entradas):
            for i in range(nb[j]):
                atraso = nk[j] + i
                linha.append(u[k - atraso, j])

        phi.append(linha)
        alvo.append(y[k])

    phi = np.array(phi, dtype=float)
    alvo = np.array(alvo, dtype=float)

    return phi, alvo, k0


def estimar_arx(y: np.ndarray,
                u: np.ndarray,
                na: int,
                nb: Union[int, List[int]],
                nk: Union[int, List[int]],
                ts: float,
                nomes_entradas: List[str],
                nome_saida: str,
                ridge_lambda: float = 0.0) -> ModeloARX:
    """
    Estima os parâmetros ARX por mínimos quadrados ou Ridge.
    """

    n_entradas = u.shape[1]
    nb, nk = tratar_nb_nk(nb, nk, n_entradas)

    phi, alvo, _ = montar_matriz_arx(y, u, na, nb, nk)

    if ridge_lambda > 0:
        I = np.eye(phi.shape[1])
        theta = np.linalg.solve(phi.T @ phi + ridge_lambda * I, phi.T @ alvo)
    else:
        theta, _, _, _ = np.linalg.lstsq(phi, alvo, rcond=None)

    a = theta[:na]

    b = []
    idx = na
    for j in range(n_entradas):
        bj = theta[idx:idx + nb[j]]
        b.append(bj)
        idx += nb[j]

    modelo = ModeloARX(
        na=na,
        nb=nb,
        nk=nk,
        theta=theta,
        a=a,
        b=b,
        ts=ts,
        nomes_entradas=nomes_entradas,
        nome_saida=nome_saida
    )

    return modelo


def prever_um_passo(modelo: ModeloARX, y: np.ndarray, u: np.ndarray):
    """
    Predição um passo à frente usando a saída real passada.
    Essa validação costuma ficar melhor que a simulação livre.
    """

    phi, alvo, k0 = montar_matriz_arx(
        y=y,
        u=u,
        na=modelo.na,
        nb=modelo.nb,
        nk=modelo.nk
    )

    y_pred = phi @ modelo.theta

    y_pred_completo = np.full_like(y, np.nan, dtype=float)
    y_pred_completo[k0:] = y_pred

    return y_pred_completo


def simular_livre(modelo: ModeloARX, y: np.ndarray, u: np.ndarray):
    """
    Simulação livre do modelo.
    Usa a própria saída simulada como realimentação.
    É mais exigente que a predição um passo à frente.
    """

    n = len(y)
    y_sim = np.zeros(n)

    maior_atraso_saida = modelo.na
    maior_atraso_entrada = max([
        modelo.nk[j] + modelo.nb[j] - 1
        for j in range(len(modelo.nb))
    ])

    k0 = max(maior_atraso_saida, maior_atraso_entrada)

    # Inicializa com a saída real
    y_sim[:k0] = y[:k0]

    for k in range(k0, n):
        valor = 0.0

        # Parte autorregressiva
        for i in range(1, modelo.na + 1):
            valor += -modelo.a[i - 1] * y_sim[k - i]

        # Parte das entradas
        for j in range(len(modelo.b)):
            for i in range(modelo.nb[j]):
                atraso = modelo.nk[j] + i
                valor += modelo.b[j][i] * u[k - atraso, j]

        y_sim[k] = valor

    return y_sim


def calcular_metricas(y_real: np.ndarray, y_est: np.ndarray) -> Dict[str, float]:
    """
    Calcula MAE, RMSE e R² ignorando NaN.
    """

    mascara = ~np.isnan(y_est)

    y_real_valid = y_real[mascara]
    y_est_valid = y_est[mascara]

    erro = y_real_valid - y_est_valid

    mae = np.mean(np.abs(erro))
    rmse = np.sqrt(np.mean(erro ** 2))

    ss_res = np.sum((y_real_valid - y_est_valid) ** 2)
    ss_tot = np.sum((y_real_valid - np.mean(y_real_valid)) ** 2)

    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


def imprimir_modelo(modelo: ModeloARX):
    """
    Imprime o modelo ARX identificado.
    """

    print("\n================================================")
    print("MODELO ARX IDENTIFICADO")
    print("================================================")

    print(f"Saída: {modelo.nome_saida}")
    print(f"Entradas: {modelo.nomes_entradas}")
    print(f"Ts = {modelo.ts:.6f} s")
    print(f"na = {modelo.na}")
    print(f"nb = {modelo.nb}")
    print(f"nk = {modelo.nk}")

    print("\nPolinômio A(z^-1):")
    texto_a = "A(z^-1) = 1"
    for i, ai in enumerate(modelo.a, start=1):
        texto_a += f" + ({ai:.6g}) z^-{i}"
    print(texto_a)

    print("\nPolinômios B(z^-1):")
    for j, bj in enumerate(modelo.b):
        nome = modelo.nomes_entradas[j]
        texto_b = f"B_{nome}(z^-1) = "
        termos = []
        for i, coef in enumerate(bj):
            atraso = modelo.nk[j] + i
            termos.append(f"({coef:.6g}) z^-{atraso}")
        texto_b += " + ".join(termos)
        print(texto_b)

    print("\nModelo:")
    print("A(z^-1)y(k) = B1(z^-1)u1(k) + B2(z^-1)u2(k) + ... + e(k)")


def imprimir_funcao_transferencia_siso(modelo: ModeloARX):
    """
    Para caso SISO, imprime G(z^-1) = B(z^-1) / A(z^-1).
    """

    if len(modelo.b) != 1:
        print("\nFunção de transferência direta impressa apenas para SISO.")
        print("Para MISO, existe uma função de transferência de cada entrada para a saída.")
        return

    print("\n================================================")
    print("FUNÇÃO DE TRANSFERÊNCIA DISCRETA - SISO")
    print("================================================")

    print("G(z^-1) = B(z^-1) / A(z^-1)")

    texto_num = ""
    termos = []
    for i, coef in enumerate(modelo.b[0]):
        atraso = modelo.nk[0] + i
        termos.append(f"({coef:.6g}) z^-{atraso}")
    texto_num = " + ".join(termos)

    texto_den = "1"
    for i, ai in enumerate(modelo.a, start=1):
        texto_den += f" + ({ai:.6g}) z^-{i}"

    print("\nNumerador:")
    print(texto_num)

    print("\nDenominador:")
    print(texto_den)

    ganho_dc = np.sum(modelo.b[0]) / (1.0 + np.sum(modelo.a))
    print(f"\nGanho estático aproximado G(1): {ganho_dc:.6g}")


def aproximar_continuo_siso(modelo: ModeloARX):
    """
    Aproxima polos e zeros contínuos usando:
    s = ln(z) / Ts

    Atenção:
    Esta não é uma conversão perfeita tipo MATLAB d2c.
    É uma aproximação por mapeamento polo-zero.
    Para controle industrial, muitas vezes é melhor ajustar um modelo
    FOPDT: K * exp(-Ls) / (tau*s + 1)
    """

    if len(modelo.b) != 1:
        print("\nAproximação contínua automática disponível apenas para SISO neste exemplo.")
        return

    print("\n================================================")
    print("APROXIMAÇÃO CONTÍNUA SISO")
    print("================================================")

    # Polos discretos a partir de A(z^-1)
    den_z = np.concatenate(([1.0], modelo.a))
    polos_z = np.roots(den_z)

    # Zeros discretos a partir do B sem o atraso puro
    b = modelo.b[0]
    if len(b) > 1:
        zeros_z = np.roots(b)
    else:
        zeros_z = np.array([])

    ts = modelo.ts

    polos_s = []
    for p in polos_z:
        if np.abs(p) > 0:
            polos_s.append(np.log(p) / ts)

    zeros_s = []
    for z in zeros_z:
        if np.abs(z) > 0:
            zeros_s.append(np.log(z) / ts)

    atraso_aprox = modelo.nk[0] * ts

    print(f"Atraso puro aproximado: L = {atraso_aprox:.6g} s")

    print("\nPolos discretos z:")
    for p in polos_z:
        print(f"{p:.6g}")

    print("\nPolos contínuos aproximados s:")
    for p in polos_s:
        print(f"{p:.6g}")

    if len(zeros_s) > 0:
        print("\nZeros contínuos aproximados s:")
        for z in zeros_s:
            print(f"{z:.6g}")
    else:
        print("\nSem zeros finitos identificados no numerador.")

    print("\nObservação:")
    print("Se os polos contínuos tiverem parte real negativa, o modelo é estável.")
    print("Se aparecer parte imaginária, há comportamento oscilatório.")


def plotar_resultados(y_real: np.ndarray,
                      y_pred: np.ndarray,
                      y_sim: np.ndarray,
                      ts: float,
                      titulo: str = "Validação do Modelo ARX"):
    """
    Plota saída real, predição um passo e simulação livre.
    """

    tempo = np.arange(len(y_real)) * ts

    plt.figure(figsize=(14, 6))
    plt.plot(tempo, y_real, label="Saída real")
    plt.plot(tempo, y_pred, label="Predição 1 passo")
    plt.plot(tempo, y_sim, label="Simulação livre", linestyle="--")
    plt.xlabel("Tempo [s]")
    plt.ylabel("Saída")
    plt.title(titulo)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    erro_pred = y_real - y_pred

    plt.figure(figsize=(14, 4))
    plt.plot(tempo, erro_pred, label="Erro - predição 1 passo")
    plt.axhline(0, linestyle="--")
    plt.xlabel("Tempo [s]")
    plt.ylabel("Erro")
    plt.title("Erro de Predição")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()



In [ ]:
# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================

def executar_identificacao():
    df = carregar_dados(ARQUIVO_DADOS)

    colunas_necessarias = [COLUNA_SAIDA] + COLUNAS_ENTRADA
    df = df[colunas_necessarias].copy()

    # Remove linhas com NaN
    df = df.dropna()

    # Converte para número
    for col in colunas_necessarias:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna()

    y = df[COLUNA_SAIDA].values.astype(float)
    u = df[COLUNAS_ENTRADA].values.astype(float)

    n = len(df)
    n_treino = int(PERC_TREINO * n)

    y_treino = y[:n_treino]
    u_treino = u[:n_treino, :]

    y_teste = y[n_treino:]
    u_teste = u[n_treino:, :]

    print("\nAmostras totais:", n)
    print("Amostras treino:", len(y_treino))
    print("Amostras teste:", len(y_teste))
    print("Número de entradas:", u.shape[1])

    modelo = estimar_arx(
        y=y_treino,
        u=u_treino,
        na=NA,
        nb=NB,
        nk=NK,
        ts=TS,
        nomes_entradas=COLUNAS_ENTRADA,
        nome_saida=COLUNA_SAIDA,
        ridge_lambda=RIDGE_LAMBDA
    )

    imprimir_modelo(modelo)
    imprimir_funcao_transferencia_siso(modelo)
    aproximar_continuo_siso(modelo)

    # Validação no conjunto de teste
    y_pred_teste = prever_um_passo(modelo, y_teste, u_teste)
    y_sim_teste = simular_livre(modelo, y_teste, u_teste)

    metricas_pred = calcular_metricas(y_teste, y_pred_teste)
    metricas_sim = calcular_metricas(y_teste, y_sim_teste)

    print("\n================================================")
    print("MÉTRICAS - PREDIÇÃO 1 PASSO")
    print("================================================")
    for k, v in metricas_pred.items():
        print(f"{k}: {v:.6g}")

    print("\n================================================")
    print("MÉTRICAS - SIMULAÇÃO LIVRE")
    print("================================================")
    for k, v in metricas_sim.items():
        print(f"{k}: {v:.6g}")

    plotar_resultados(
        y_real=y_teste,
        y_pred=y_pred_teste,
        y_sim=y_sim_teste,
        ts=TS,
        titulo="Validação do Modelo ARX no Conjunto de Teste"
    )

    return modelo


if __name__ == "__main__":
    modelo_identificado = executar_identificacao()